In [1]:
import shap
from scipy.stats import spearmanr
from itertools import combinations
import os
import pickle
import numpy as np
import inspect
import shap


/hpc/uu_inf_aidsaitfl/miniforge3/envs/dimaf2/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_shap_dict(dir, type=None):
    if type is not None:
        shap_results_fold_dir = os.path.join(dir, f'shap_all_test_{type}.pkl')
    elif type is None:
        shap_results_fold_dir = os.path.join(dir, f'shap_all_test.pkl')
    shap_dict = pickle.load(open(shap_results_fold_dir, 'rb'))     
    shap_values = np.sum(shap_dict['shap values'], axis=2)
    feature_names = list(shap_dict['Feature names'])
    return shap_values, feature_names

In [28]:
# For each fold
type='kirc'
dict = {
    'brca': ['64', '128', '192', '256', '320', '384', '448', 'seed_1', '576', '640', '704'],
    'blca': ['64', '128', '192', 'seed_1'],
    'luad': ['64', '128', '192', '256', 'seed_1'],
    'kirc': ['64', '128', 'seed_1']
}
dir = f'../results/ablations/dss_survival_{type}/DIMAFx/'
for repr_type in ['modal', 'post_attn', 'post_attn_av']:
    print(f"Representation type: {repr_type}")
    for i in range(5):
        print("Fold: ", i)
        for combo in combinations(dict[type], 2):
            print(f"Comparing {combo[0]} vs {combo[1]}")
            results = {}
            fold_dir = os.path.join(dir, f'Fold_{i}/post_training/shap/{repr_type}')
            shap_values_1, _ = get_shap_dict(fold_dir, type=f"{combo[0]}")
            shap_values_2, _ = get_shap_dict(fold_dir, type=f"{combo[1]}")

            importance_1 = np.mean(np.abs(shap_values_1), axis=0)
            importance_2 = np.mean(np.abs(shap_values_2), axis=0)

            # Spearman rank correlation
            r, p = spearmanr(importance_1, importance_2)
            print(f"Spearman rank correlation: {r:.3f}, p={p:.3e}")
            if p >= 0.05:
                print(f"Correlation {r} is not significant (p={p:.3e})")
            if r < min1:
                min1 = r

            # [n_test_samples, n_feats] after summing over embed dim
            importance_1_per_sample = np.abs(shap_values_1)
            importance_2_per_sample = np.abs(shap_values_2)

            # Correlation per sample
            correlations = []
            for s in range(importance_1_per_sample.shape[0]):
                r, p = spearmanr(importance_1_per_sample[s], importance_2_per_sample[s])
                correlations.append(r)
                if p >= 0.05:
                    print(f"Correlation for sample {s} is not significant (p={p:.3e})")
                    print()
                if r < min3:
                    min3 = r
                

            correlations = np.array(correlations)
            print(f"Mean Spearman per sample: {np.mean(np.abs(correlations)):.3f} ± {np.std(np.abs(correlations)):.3f}")
            print(f"Min: {np.min(correlations):.3f}, Max: {np.max(correlations):.3f}")
            if np.mean(np.abs(correlations)) < min2:
                min2 = np.mean(np.abs(correlations))

print(f"Minimum Spearman rank correlation across all combos and folds: {min1:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min2:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min3:.3f}")



Representation type: modal
Fold:  0
Comparing 64 vs 128
Spearman rank correlation: 0.997, p=8.052e-73
Mean Spearman per sample: 0.982 ± 0.011
Min: 0.933, Max: 0.994
Comparing 64 vs seed_1
Spearman rank correlation: 0.996, p=4.030e-69
Mean Spearman per sample: 0.979 ± 0.014
Min: 0.920, Max: 0.995
Comparing 128 vs seed_1
Spearman rank correlation: 0.998, p=4.605e-82
Mean Spearman per sample: 0.993 ± 0.004
Min: 0.982, Max: 0.998
Fold:  1
Comparing 64 vs 128
Spearman rank correlation: 0.991, p=1.013e-57
Mean Spearman per sample: 0.985 ± 0.008
Min: 0.959, Max: 0.997
Comparing 64 vs seed_1
Spearman rank correlation: 0.984, p=9.468e-50
Mean Spearman per sample: 0.974 ± 0.015
Min: 0.923, Max: 0.994
Comparing 128 vs seed_1
Spearman rank correlation: 0.997, p=1.933e-72
Mean Spearman per sample: 0.989 ± 0.007
Min: 0.959, Max: 0.997
Fold:  2
Comparing 64 vs 128
Spearman rank correlation: 0.994, p=3.461e-62
Mean Spearman per sample: 0.987 ± 0.007
Min: 0.967, Max: 0.998
Comparing 64 vs seed_1
Spearm

In [29]:
# For each fold
type='brca'
dir = f'../results/ablations/dss_survival_{type}/DIMAFx/'
min1 = 1
min2 = 1
min3 = 1
for repr_type in ['modal', 'post_attn']:
    for i in range(5):
        for combo in combinations([1, 2, 3, 4, 5], 2):
            print("Combo: ", combo)
            
            print("Fold: ", i)
            results = {}
            fold_dir = os.path.join(dir, f'Fold_{i}/post_training/shap/{repr_type}')
            shap_values_1, _ = get_shap_dict(fold_dir, type=f"seed_{combo[0]}")
            shap_values_2, _ = get_shap_dict(fold_dir, type=f"seed_{combo[1]}")

            importance_1 = np.mean(np.abs(shap_values_1), axis=0)
            importance_2 = np.mean(np.abs(shap_values_2), axis=0)

            # Spearman rank correlation
            r, p = spearmanr(importance_1, importance_2)
            print(f"Spearman rank correlation: {r:.3f}, p={p:.3e}")
            if p >= 0.05:
                print(f"Correlation {r} is not significant (p={p:.3e})")
            if r < min1:
                min1 = r

            # [n_test_samples, n_feats] after summing over embed dim
            importance_1_per_sample = np.abs(shap_values_1)
            importance_2_per_sample = np.abs(shap_values_2)

            # Correlation per sample
            correlations = []
            for s in range(importance_1_per_sample.shape[0]):
                r, p = spearmanr(importance_1_per_sample[s], importance_2_per_sample[s])
                correlations.append(r)
                if p >= 0.05:
                    print(f"Correlation for sample {s} is not significant (p={p:.3e})")
                    print()
                if r < min3:
                    min3 = r
                

            correlations = np.array(correlations)
            print(f"Mean Spearman per sample: {np.mean(np.abs(correlations)):.3f} ± {np.std(np.abs(correlations)):.3f}")
            print(f"Min: {np.min(correlations):.3f}, Max: {np.max(correlations):.3f}")
            if np.mean(np.abs(correlations)) < min2:
                min2 = np.mean(np.abs(correlations))

print(f"Minimum Spearman rank correlation across all combos and folds: {min1:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min2:.3f}")
print(f"Minimum Spearman rank correlation per sample across all combos and folds: {min3:.3f}")


Combo:  (1, 2)
Fold:  0
Spearman rank correlation: 0.999, p=5.631e-85
Mean Spearman per sample: 0.994 ± 0.007
Min: 0.953, Max: 1.000
Combo:  (1, 3)
Fold:  0
Spearman rank correlation: 0.998, p=1.080e-81
Mean Spearman per sample: 0.994 ± 0.007
Min: 0.953, Max: 1.000
Combo:  (1, 4)
Fold:  0
Spearman rank correlation: 0.999, p=5.631e-85
Mean Spearman per sample: 0.997 ± 0.002
Min: 0.990, Max: 0.999
Combo:  (1, 5)
Fold:  0
Spearman rank correlation: 0.995, p=1.085e-66
Mean Spearman per sample: 0.995 ± 0.003
Min: 0.982, Max: 0.999
Combo:  (2, 3)
Fold:  0
Spearman rank correlation: 0.999, p=1.937e-86
Mean Spearman per sample: 0.997 ± 0.001
Min: 0.991, Max: 0.999
Combo:  (2, 4)
Fold:  0
Spearman rank correlation: 0.999, p=5.794e-87
Mean Spearman per sample: 0.993 ± 0.010
Min: 0.950, Max: 1.000
Combo:  (2, 5)
Fold:  0
Spearman rank correlation: 0.998, p=4.606e-76
Mean Spearman per sample: 0.991 ± 0.011
Min: 0.946, Max: 0.999
Combo:  (3, 4)
Fold:  0
Spearman rank correlation: 0.999, p=1.937e-86